In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os
import re

class WorkSafetyEvidence:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.data = None
        self.embeddings = None
        self.merge_map = {
            0: 'Medical/Illness', 1: 'Vehicle/Transport', 2: 'Caught/Equipment',
            3: 'Other/Animal', 4: 'Falls', 5: 'Falls', 6: 'Caught/Equipment',
            7: 'Caught/Equipment', 8: 'Chemical/Exposure', 9: 'Chemical/Exposure',
            10: 'Chemical/Exposure', 11: 'Medical/Illness', 12: 'Medical/Illness',
            13: 'Electrical', 14: 'Medical/Illness'
        }

    def clean_text(self, t):
        t = str(t).lower()
        t = re.sub(r'[^a-z\s]', '', t)
        t = re.sub(r'\s+', ' ', t)
        return t.strip()

    def load_data(self, file_path):
        print("Loading historical accident data...")
        df = pd.read_excel(file_path)
        df = df.drop_duplicates(subset=['abstract', 'event_type'])

        text_columns = ['abstract', 'event_keyword', 'main', 'description']
        for col in text_columns:
            if col in df.columns:
                df[col] = df[col].fillna('').astype(str)

        df['text_for_bert'] = (
            df['abstract'].fillna('') + ' ' +
            df['event_keyword'].fillna('') + ' ' +
            df.get('description', '')
        ).apply(self.clean_text)

        df['merged_event'] = df['event_type'].map(self.merge_map)

        essential_columns = [
            'text_for_bert', 'abstract', 'event_type', 'degree_of_inj_x',
            'event_keyword', 'merged_event'
        ]
        self.data = df[essential_columns]

        print(f"Loaded {len(self.data)} past accidents")
        print("Category distribution:")
        print(self.data['merged_event'].value_counts())
        return self.data

    def create_embeddings(self):
        print("Creating embeddings from accident data...")
        texts = self.data['text_for_bert'].tolist()
        self.embeddings = self.model.encode(texts, show_progress_bar=True)
        print("Accident embeddings ready!")
        return self.embeddings

    def save_system(self, save_folder="safety_system"):
        if not os.path.exists(save_folder):
            os.makedirs(save_folder)

        with open(f'{save_folder}/embeddings.pkl', 'wb') as f:
            pickle.dump(self.embeddings, f)

        essential_columns = ['abstract', 'event_type', 'degree_of_inj_x', 'merged_event', 'event_keyword']
        essential_data = self.data[essential_columns]
        essential_data.to_pickle(f'{save_folder}/accident_data.pkl')

        print(f"System saved to '{save_folder}' folder!")

    def load_system(self, save_folder="safety_system"):
        try:
            with open(f'{save_folder}/embeddings.pkl', 'rb') as f:
                self.embeddings = pickle.load(f)

            self.data = pd.read_pickle(f'{save_folder}/accident_data.pkl')

            print(f"System loaded from '{save_folder}'!")
            print(f"{len(self.data)} incidents ready")
            return True
        except FileNotFoundError:
            print("No saved system found. Please create embeddings first.")
            return False

    def find_similar_accidents(self, planned_work, top_k=5):
        if self.embeddings is None:
            return "Please create embeddings first!"

        clean_query = self.clean_text(planned_work)
        work_embedding = self.model.encode([clean_query])
        similarities = cosine_similarity(work_embedding, self.embeddings)[0]
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        evidence = []
        for idx in top_indices:
            accident = self.data.iloc[idx]
            evidence.append({
                'similarity': round(similarities[idx], 3),
                'past_accident': accident['abstract'],
                'accident_type': accident.get('event_type', 'N/A'),
                'event_category': accident.get('merged_event', 'N/A'),
                'severity': accident.get('degree_of_inj_x', 'N/A'),
                'keywords': accident.get('event_keyword', '')
            })
        return evidence

In [ ]:
def setup_system():
    safety_system = WorkSafetyEvidence()
    safety_system.load_data("Injury Severity.xlsx")
    safety_system.create_embeddings()
    safety_system.save_system()
    return safety_system

safety_system = setup_system()
print("One-time setup complete! Now use fast loading.")

Loading historical accident data...
Loaded 49236 past accidents
Category distribution:
merged_event
Caught/Equipment     14145
Falls                11227
Medical/Illness      10781
Vehicle/Transport     9936
Electrical            1575
Chemical/Exposure     1428
Other/Animal           144
Name: count, dtype: int64
Creating embeddings from accident data...


Batches:   0%|          | 0/1539 [00:00<?, ?it/s]

Accident embeddings ready!
System saved to 'safety_system' folder!
One-time setup complete! Now use fast loading.


In [ ]:
def load_system_fast():
    safety_system = WorkSafetyEvidence()
    if safety_system.load_system():
        print("System ready for instant queries!")
        return safety_system
    else:
        return None

system = load_system_fast()

System loaded from 'safety_system'!
49236 incidents ready
System ready for instant queries!


In [ ]:
test_query = "The crew will install prefabricated steel roof trusses on the third level of the commercial building. Two mobile cranes and chain hoists will be used to lift materials into position. Workers will secure the trusses using impact wrenches and welding torches while standing on temporary scaffolds and roof beams. The operation is scheduled for late afternoon when wind speeds typically increase. Fall-protection harnesses and guardrails have been provided, but footing is expected to be slippery due to residual moisture from morning rain"

if system:
    print(f"Testing with construction safety query...")

    results = system.find_similar_accidents(test_query, top_k=3)

    for i, result in enumerate(results, 1):
        print(f"{i}. Similarity Score: {result['similarity']}")
        print(f"   Event Category: {result['event_category']}")
        print(f"   Accident Type: {result['accident_type']}")
        print(f"   Severity Level: {result['severity']}")
        print(f"   Keywords: {result['keywords']}")
        print(f"   Past Accident Summary: {result['past_accident'][:200]}...")
        print("-" * 80)
else:
    print("System not loaded. Please run one-time setup first.")

Testing with construction safety query...
1. Similarity Score: 0.6669999957084656
   Event Category: Caught/Equipment
   Accident Type: 2
   Severity Level: 2
   Keywords: BRACING,COLLAPSE,FRACTURE,INSTALLING,RESIDENTIAL CONSTRUCTION,ROOF COLLAPSE,TRUSS
   Past Accident Summary: At 1:30 p.m. on October 7, 2019, a crew of five carpenters were installing a roof truss system when the trusses collapsed.  The employer did not ensure adequate ground and truss bracing were installed...
--------------------------------------------------------------------------------
2. Similarity Score: 0.6119999885559082
   Event Category: Falls
   Accident Type: 5
   Severity Level: 2
   Keywords: BOOM,BRACING,CARPENTER,CONSTRUCTION,FALL,FALL PROTECTION,FRACTURE,INSTALLING,LEG,ROOF,ROOF DECKING,SCISSOR LIFT,STRUCTURAL COLLAPSE,THIGH,TRUSS,UNSECURED
   Past Accident Summary: At 2:45 p.m. on October 9, 2018, Employee #1, the company owner, and three cowor kers, employed by a construction framing company, were 

In [ ]:
test_query = "Our electrical team needs to install the main power distribution panel in the basement of the commercial complex. They'll be working with high-voltage cables and circuit breakers. We've scheduled a main power shutdown from 10 AM to 12 PM for safe installation."


if system:
    print(f"Testing with construction safety query...")

    results = system.find_similar_accidents(test_query, top_k=5)

    for i, result in enumerate(results, 1):
        print(f"{i}. Similarity Score: {result['similarity']}")
        print(f"   Event Category: {result['event_category']}")
        print(f"   Accident Type: {result['accident_type']}")
        print(f"   Severity Level: {result['severity']}")
        print(f"   Keywords: {result['keywords']}")
        print(f"   Past Accident Summary: {result['past_accident'][:200]}...")
        print("-" * 80)
else:
    print("System not loaded. Please run one-time setup first.")

Testing with construction safety query...
1. Similarity Score: 0.49300000071525574
   Event Category: Electrical
   Accident Type: 13
   Severity Level: 1
   Keywords: CONSTRUCTION,CONTACT,ELECTRIC SHOCK,ELECTROCUTED,FALL,FALL PROTECTION,INSTALLING,OVERHEAD POWER LINE
   Past Accident Summary: At 1:50 p.m. on March 18, 2020, Employee #1 was working at a multiemployer project, the renovation of a commercial structure.  He was installing gutters when he made contact with overhead power lines....
--------------------------------------------------------------------------------
2. Similarity Score: 0.49000000953674316
   Event Category: Electrical
   Accident Type: 13
   Severity Level: 2
   Keywords: BURN,CONSTRUCTION,ELEC UTILITY WORK,ELECTRIC SHOCK,ELECTRICAL WORK,LOCKOUT/TAGOUT,POWER LINE WORKER,POWER LINES,UNGUARDED LIVE PARTS
   Past Accident Summary: At 7:10 p.m. on June 6, 2018, Employee #1, employed by an electrical services co mpany, was working on electrical power lines following

In [ ]:
import numpy as np

def calculate_precision_at_k(all_results, k):
    """Precision@K = Relevant documents in top K / K"""
    precisions = []
    for result in all_results:
        relevant_in_top_k = sum(result['relevance'][:k])
        precision = relevant_in_top_k / k
        precisions.append(precision)
    return np.mean(precisions)

def calculate_recall_at_k(all_results, k):
    """Recall@K = Relevant documents in top K / Total relevant documents"""
    recalls = []
    for result in all_results:
        relevant_in_top_k = sum(result['relevance'][:k])
        total_relevant = max(1, sum(result['relevance']))
        recall = relevant_in_top_k / total_relevant
        recalls.append(recall)
    return np.mean(recalls)

def calculate_f1_at_k(all_results, k):
    """F1@K = 2 * (P@K * R@K) / (P@K + R@K)"""
    p = calculate_precision_at_k(all_results, k)
    r = calculate_recall_at_k(all_results, k)
    if p + r == 0:
        return 0
    return 2 * (p * r) / (p + r)

def calculate_mrr(all_results):
    """MRR = Mean of 1/rank of first relevant document"""
    reciprocal_ranks = []
    for result in all_results:
        for rank, rel in enumerate(result['relevance'], 1):
            if rel == 1:
                reciprocal_ranks.append(1.0 / rank)
                break
        else:
            reciprocal_ranks.append(0.0)
    return np.mean(reciprocal_ranks)

def calculate_map(all_results):
    """MAP = Mean Average Precision"""
    average_precisions = []
    for result in all_results:
        precision_scores = []
        num_relevant = 0
        for i, rel in enumerate(result['relevance'], 1):
            if rel == 1:
                num_relevant += 1
                precision_scores.append(num_relevant / i)
        if precision_scores:
            ap = np.mean(precision_scores)
            average_precisions.append(ap)
        else:
            average_precisions.append(0.0)
    return np.mean(average_precisions)

def calculate_ndcg(all_results, k=10):
    """nDCG@K = Normalized Discounted Cumulative Gain"""
    ndcg_scores = []
    for result in all_results:
        dcg = 0.0
        for i, rel in enumerate(result['relevance'][:k], 1):
            dcg += rel / np.log2(i + 1)

        ideal_relevance = sorted(result['relevance'], reverse=True)[:k]
        idcg = sum(rel / np.log2(i + 1) for i, rel in enumerate(ideal_relevance, 1))

        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcg_scores.append(ndcg)

    return np.mean(ndcg_scores)

def proper_evaluation(system, test_queries):
    """Proper evaluation using standard information retrieval metrics"""
    all_results = []

    print("PROPER EVALUATION METRICS")
    print("=" * 70)

    for query_info in test_queries:
        query = query_info['query']
        expected_category = query_info['expected_category']

        results = system.find_similar_accidents(query, top_k=10)

        relevance_scores = [1 if r['event_category'] == expected_category else 0 for r in results]
        similarities = [r['similarity'] for r in results]

        all_results.append({
            'query': query,
            'expected': expected_category,
            'relevance': relevance_scores,
            'similarities': similarities,
            'retrieved_categories': [r['event_category'] for r in results]
        })

    precisions = {}
    for k in [1, 3, 5, 10]:
        precisions[k] = calculate_precision_at_k(all_results, k)

    recalls = {}
    for k in [1, 3, 5, 10]:
        recalls[k] = calculate_recall_at_k(all_results, k)

    mrr = calculate_mrr(all_results)
    map_score = calculate_map(all_results)
    ndcg = calculate_ndcg(all_results)

    f1_scores = {}
    for k in [1, 3, 5, 10]:
        f1_scores[k] = calculate_f1_at_k(all_results, k)

    print("\n" + "="*70)
    print("PRECISION, RECALL & F1 SCORES")
    print("="*70)

    print(f"{'METRIC':<12} {'VALUE':<8} {'INTERPRETATION'}")
    print("-" * 70)

    for k in [1, 3, 5, 10]:
        print(f"P@{k}:<8 {precisions[k]:.4f}    % of top-{k} results that are relevant")
        print(f"R@{k}:<8 {recalls[k]:.4f}    % of relevant results found in top-{k}")
        print(f"F1@{k}:<8 {f1_scores[k]:.4f}    Balance between P@{k} and R@{k}")
        print("-" * 70)

    print("\n" + "="*70)
    print("RANKING METRICS")
    print("="*70)

    print(f"{'METRIC':<12} {'VALUE':<8} {'INTERPRETATION'}")
    print("-" * 70)
    print(f"MRR:<11 {mrr:.4f}    Mean Reciprocal Rank (first relevant result)")
    print("-" * 70)
    print(f"MAP:<11 {map_score:.4f}    Mean Average Precision (overall ranking)")
    print("-" * 70)
    print(f"nDCG:<10 {ndcg:.4f}    Normalized Discounted Cumulative Gain")
    print("-" * 70)

    print("\n" + "="*70)
    print("PERFORMANCE SUMMARY")
    print("="*70)

    # Key metrics summary
    print(f"Key Precision Metrics:")
    print(f"  • P@1:  {precisions[1]:.4f} - First result relevance")
    print(f"  • P@3:  {precisions[3]:.4f} - Top 3 results relevance")
    print(f"  • P@5:  {precisions[5]:.4f} - Top 5 results relevance")
    print(f"  • MRR:  {mrr:.4f} - Speed to first relevant result")
    print(f"  • MAP:  {map_score:.4f} - Overall ranking quality")

    return {
        'precisions': precisions,
        'recalls': recalls,
        'f1_scores': f1_scores,
        'mrr': mrr,
        'map': map_score,
        'ndcg': ndcg
    }

# Test queries for your system
test_queries = [
    {
        'query': "Worker fell from roof while installing trusses and suffered multiple injuries",
        'expected_category': 'Falls'
    },
    {
        'query': "Electrician received severe shock from live electrical panel causing cardiac arrest",
        'expected_category': 'Electrical'
    },
    {
        'query': "Chemical operator exposed to toxic fumes causing respiratory failure and lung damage",
        'expected_category': 'Chemical/Exposure'
    },
    {
        'query': "Factory worker's hand caught in machinery resulting in traumatic amputation",
        'expected_category': 'Caught/Equipment'
    },
    {
        'query': "Construction worker struck by reversing vehicle at site causing critical injuries",
        'expected_category': 'Vehicle/Transport'
    },
    {
        'query': "Employee collapsed from heat stroke while working in extreme temperatures",
        'expected_category': 'Medical/Illness'
    }
]

# Run the evaluation
print("Running evaluation on your evidence retrieval system...")
metrics = proper_evaluation(system, test_queries)

Running evaluation on your evidence retrieval system...
PROPER EVALUATION METRICS

PRECISION, RECALL & F1 SCORES
METRIC       VALUE    INTERPRETATION
----------------------------------------------------------------------
P@1:<8 0.8333    % of top-1 results that are relevant
R@1:<8 0.0912    % of relevant results found in top-1
F1@1:<8 0.1644    Balance between P@1 and R@1
----------------------------------------------------------------------
P@3:<8 0.8889    % of top-3 results that are relevant
R@3:<8 0.3027    % of relevant results found in top-3
F1@3:<8 0.4516    Balance between P@3 and R@3
----------------------------------------------------------------------
P@5:<8 0.8667    % of top-5 results that are relevant
R@5:<8 0.4851    % of relevant results found in top-5
F1@5:<8 0.6220    Balance between P@5 and R@5
----------------------------------------------------------------------
P@10:<8 0.8833    % of top-10 results that are relevant
R@10:<8 1.0000    % of relevant results found in